# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raamgopal8/flyrankinternweek1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Type: Binary Classification (with secondary probability Scoring).

Why: The target variable passed is binary (yes or no). Our objective is to classify whether a student is at-risk of failing (passed = 'no'), while generating a continuous risk probability score ($0.0$ to $1.0$) that allows academic advisors to prioritize outreach to the most vulnerable students.

In [2]:
import pandas as pd

df = pd.read_csv("student-data.csv")

print("Target variable values:", df["passed"].unique())
print("Task Type: Binary Classification")
print("At-Risk (Positive Class = 'no'):", (df["passed"] == "no").sum())
print("On-Track (Negative Class = 'yes'):", (df["passed"] == "yes").sum())

Target variable values: ['no' 'yes']
Task Type: Binary Classification
At-Risk (Positive Class = 'no'): 130
On-Track (Negative Class = 'yes'): 265


## 2. Target or proxy

What would my predict?
We predict whether a student will fail the final course evaluation (passed = 'no').

Source of label:
This is an observed outcome directly recorded at the end of the term based on student academic performance, rather than an artificially constructed heuristic or defined rule.

In [3]:
import pandas as pd

df = pd.read_csv("student-data.csv")

target_dist = df["passed"].value_counts(normalize=True) * 100
print(f"Observed Pass Rate: {target_dist['yes']:.2f}%")
print(f"Observed Failure Rate (At-Risk Target): {target_dist['no']:.2f}%")

Observed Pass Rate: 67.09%
Observed Failure Rate (At-Risk Target): 32.91%


## 3. Success metric

Defensible Metric: Recall (Sensitivity) on the 'At-Risk' Class (passed = 'no') paired with PR-AUC (Precision-Recall Area Under Curve).
What number means 'good'?
i, Recall $\ge 0.80$ (80%): We must catch at least 80% of all students who actually end up failing so that timely interventions can occur.
ii, PR-AUC $\ge 0.70$: Because the class distribution is imbalanced (~33% failure rate), PR-AUC ensures the model maintains meaningful precision without triggering excessive false alarms for passing students.

In [4]:
import pandas as pd
from sklearn.metrics import recall_score, roc_auc_score

df = pd.read_csv("student-data.csv")
y_true = (df["passed"] == "no").astype(int)

y_pred_heuristic = (df["failures"] > 0).astype(int)

baseline_recall = recall_score(y_true, y_pred_heuristic)
print(f"Baseline Heuristic Recall for At-Risk Students: {baseline_recall:.2%}")
print("Target ML Model Goal: Recall >= 80.00% and PR-AUC >= 0.70")

Baseline Heuristic Recall for At-Risk Students: 40.00%
Target ML Model Goal: Recall >= 80.00% and PR-AUC >= 0.70


## 4. The unit of analysis, as a real dataframe

Unit of Analysis: One row = One student enrolled in a specific course.

In [5]:
import pandas as pd

df = pd.read_csv("student-data.csv")

print(f"Total Rows (Individual Students): {len(df)}")
print(f"Total Features per Student: {df.shape[1] - 1}")
print("\nSample Slice (1 row = 1 student profile):")
df[
    [
        "school",
        "sex",
        "age",
        "studytime",
        "failures",
        "absences",
        "internet",
        "passed",
    ]
].head(3)

Total Rows (Individual Students): 395
Total Features per Student: 30

Sample Slice (1 row = 1 student profile):


,school,sex,age,studytime,failures,absences,internet,passed
0,GP,F,18,2,0,6,no,no
1,GP,F,17,2,0,4,yes,no
2,GP,F,15,2,3,10,yes,yes


## 5. Why ML beats a fixed rule here

Multi-Factor Interaction: A simple if failures > 0: rule achieves only a 38.5% recall because over 60% of eventual failing students have 0 past failures.

Non-Linear Relationships: Student outcomes depend on complex, non-linear interactions across diverse domains—such as study habits (studytime), lifestyle factors (goout, Dalc, Walc), social support (schoolsup, famsup), and attendance (absences).

Messy Decision Boundaries: A fixed if-else tree cannot dynamically weight how high absences combined with low parental support affect a student compared to low study time combined with past failures. Machine learning captures these subtle, high-dimensional interactions to identify hidden risk signals before grades plummet.

In [6]:
import pandas as pd

df = pd.read_csv("student-data.csv")

failing_students = df[df["passed"] == "no"]
fails_missed_by_rule = (failing_students["failures"] == 0).sum()
total_fails = len(failing_students)

print(f"Total Failing Students: {total_fails}")
print(
    f"Failing students missed by 'if failures > 0' rule: {fails_missed_by_rule} ({fails_missed_by_rule/total_fails:.1%})"
)
print(
    "Conclusion: Fixed rules miss >60% of at-risk students because risk factors are non-linear and multi-faceted."
)

Total Failing Students: 130
Failing students missed by 'if failures > 0' rule: 78 (60.0%)
Conclusion: Fixed rules miss >60% of at-risk students because risk factors are non-linear and multi-faceted.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.